# JachaiX — Chunker & Qdrant Indexer
Reads all articles from `corpus/raw/`, splits them into 300-700 token chunks,
calls the Embedder API to get vectors, and stores everything in Qdrant.

**Run order:** Cell 1 → 2 → 3 → 4 → 5

**Chunk metadata stored per chunk:**
- `chunk_text`, `source_article_title`, `source_url`, `published_date`
- `chunk_index`, `language`, `source_name`, `reliability_score`

In [ ]:
## Cell 1 — Imports & Config
import json, uuid, time, re
from pathlib import Path
import requests

RAW_DIR     = Path("E:/jachaix/corpus/raw")
EMBEDDER_URL = "http://localhost:5002/embed/text"
QDRANT_URL   = "http://localhost:6333"
COLLECTION   = "knowledge_base"

# Chunking settings (senior's recommendation: 300-700 tokens)
# Approximation: 1 token ≈ 4 chars for mixed Bangla/English
CHUNK_TARGET_CHARS = 1600   # ~400 tokens
CHUNK_MAX_CHARS    = 2800   # ~700 tokens
CHUNK_MIN_CHARS    = 200    # ~50 tokens — discard smaller chunks

def count_articles():
    return len(list(RAW_DIR.glob("*.json")))

# Verify services are reachable
try:
    r = requests.get(f"{QDRANT_URL}/collections", timeout=5)
    print(f"Qdrant: ✅ reachable ({r.status_code})")
except Exception as e:
    print(f"Qdrant: ❌ {e}")

try:
    r = requests.post(EMBEDDER_URL, json={"text": "test"}, timeout=10)
    print(f"Embedder: ✅ reachable ({r.status_code})")
except Exception as e:
    print(f"Embedder: ❌ {e}")

print(f"\nArticles in corpus/raw: {count_articles()}")

In [ ]:
## Cell 2 — Create Qdrant Collection
# Creates the "knowledge_base" collection if it doesn't already exist.
# Vector size 768 = sentence-transformers paraphrase-multilingual-mpnet-base-v2 output.

VECTOR_SIZE = 768

# Check if collection already exists
r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", timeout=5)
if r.status_code == 200:
    info = r.json()
    count = info.get("result", {}).get("vectors_count", 0)
    print(f"Collection '{COLLECTION}' already exists with {count} vectors.")
    print("To re-index from scratch, delete it first:")
    print(f"  requests.delete('{QDRANT_URL}/collections/{COLLECTION}')")
else:
    # Create collection
    payload = {
        "vectors": {
            "size": VECTOR_SIZE,
            "distance": "Cosine"
        }
    }
    r = requests.put(f"{QDRANT_URL}/collections/{COLLECTION}", json=payload, timeout=10)
    if r.status_code in (200, 201):
        print(f"✅ Collection '{COLLECTION}' created (size={VECTOR_SIZE}, distance=Cosine)")
    else:
        print(f"❌ Failed to create collection: {r.status_code} — {r.text}")

In [ ]:
## Cell 3 — Chunking Functions
# Splits article text into semantic paragraph-boundary chunks (300-700 tokens).

def split_into_chunks(text: str, title: str) -> list[str]:
    """
    Split text into chunks at paragraph boundaries.
    Target: CHUNK_TARGET_CHARS chars (~400 tokens).
    Never split a paragraph mid-sentence.
    Always prepend article title to first chunk for context.
    """
    # Normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text.strip())

    # Split on paragraph boundaries (double newline or sentence-ending punctuation + space)
    paragraphs = re.split(r'\n\n+', text)
    if len(paragraphs) == 1:
        # Flat text — split on sentence boundaries instead
        paragraphs = re.split(r'(?<=[।.!?])\s+', text)

    chunks = []
    current = f"শিরোনাম: {title}\n\n" if title else ""  # prepend title to first chunk

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # If adding this paragraph keeps us under max → add it
        if len(current) + len(para) + 2 <= CHUNK_MAX_CHARS:
            current += para + " "
        else:
            # Save current chunk if it meets minimum size
            if len(current.strip()) >= CHUNK_MIN_CHARS:
                chunks.append(current.strip())
            # Start new chunk with this paragraph
            # If paragraph itself is too long → split it by sentences
            if len(para) > CHUNK_MAX_CHARS:
                sentences = re.split(r'(?<=[।.!?])\s+', para)
                current = ""
                for sent in sentences:
                    if len(current) + len(sent) + 1 <= CHUNK_MAX_CHARS:
                        current += sent + " "
                    else:
                        if len(current.strip()) >= CHUNK_MIN_CHARS:
                            chunks.append(current.strip())
                        current = sent + " "
            else:
                current = para + " "

    # Don't forget the last chunk
    if len(current.strip()) >= CHUNK_MIN_CHARS:
        chunks.append(current.strip())

    return chunks


def embed_text(text: str) -> list[float] | None:
    """Call the embedder service to get a vector for a text chunk."""
    try:
        r = requests.post(EMBEDDER_URL, json={"text": text}, timeout=30)
        r.raise_for_status()
        data = r.json()
        # Handle both {"embedding": [...]} and {"embeddings": [[...]]} formats
        if "embedding" in data:
            return data["embedding"]
        elif "embeddings" in data:
            return data["embeddings"][0]
        else:
            return list(data.values())[0]
    except Exception as e:
        print(f"    [EMBED ERROR] {e}")
        return None


def upload_chunk_to_qdrant(chunk_id: str, vector: list[float], payload: dict) -> bool:
    """Upload a single chunk+vector to Qdrant."""
    body = {
        "points": [{
            "id": chunk_id,
            "vector": vector,
            "payload": payload
        }]
    }
    try:
        r = requests.put(
            f"{QDRANT_URL}/collections/{COLLECTION}/points",
            json=body, timeout=15
        )
        return r.status_code in (200, 201)
    except Exception as e:
        print(f"    [QDRANT ERROR] {e}")
        return False


print("Chunking functions defined ✅")
print(f"Target chunk size : ~{CHUNK_TARGET_CHARS} chars (~400 tokens)")
print(f"Max chunk size    : ~{CHUNK_MAX_CHARS} chars (~700 tokens)")
print(f"Min chunk size    : {CHUNK_MIN_CHARS} chars (discard smaller)")

In [ ]:
## Cell 4 — Process All Articles → Embed → Store in Qdrant

all_files = sorted(RAW_DIR.glob("*.json"))
print(f"Processing {len(all_files)} articles...\n")

total_chunks   = 0
total_uploaded = 0
total_failed   = 0
skipped        = 0

for file_idx, filepath in enumerate(all_files):
    with open(filepath, encoding="utf-8") as f:
        article = json.load(f)

    content  = article.get("content", "").strip()
    title    = article.get("title", "")
    url      = article.get("url", "")
    source   = article.get("source", "unknown")
    language = article.get("language", "bn")
    pub_date = article.get("published_date", "")
    reliability = article.get("reliability_score", 0.75)

    if len(content) < CHUNK_MIN_CHARS:
        skipped += 1
        continue

    chunks = split_into_chunks(content, title)
    if not chunks:
        skipped += 1
        continue

    print(f"[{file_idx+1}/{len(all_files)}] {source} | {len(chunks)} chunks | {title[:50]}")

    for chunk_idx, chunk_text in enumerate(chunks):
        vector = embed_text(chunk_text)
        if vector is None:
            total_failed += 1
            continue

        chunk_id = str(uuid.uuid4())
        payload = {
            "chunk_text":          chunk_text,
            "source_article_title": title,
            "source_url":          url,
            "published_date":      pub_date,
            "chunk_index":         chunk_idx,
            "language":            language,
            "source_name":         source,
            "reliability_score":   reliability,
        }

        ok = upload_chunk_to_qdrant(chunk_id, vector, payload)
        if ok:
            total_chunks += 1
            total_uploaded += 1
        else:
            total_failed += 1

        time.sleep(0.1)  # small delay to avoid overwhelming the embedder

print(f"\n{'='*50}")
print(f"DONE")
print(f"Articles processed : {len(all_files) - skipped}")
print(f"Articles skipped   : {skipped} (too short)")
print(f"Total chunks       : {total_chunks}")
print(f"Uploaded to Qdrant : {total_uploaded}")
print(f"Failed             : {total_failed}")

In [ ]:
## Cell 5 — Verify Qdrant Index

r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", timeout=5)
info = r.json().get("result", {})
vectors_count = info.get("vectors_count", 0)
points_count  = info.get("points_count", 0)

print("=" * 50)
print("QDRANT INDEX VERIFICATION")
print("=" * 50)
print(f"Collection      : {COLLECTION}")
print(f"Vectors stored  : {vectors_count}")
print(f"Points stored   : {points_count}")
print()

# Quick test — search for a sample query to confirm retrieval works
test_query = "Bangladesh fake news misinformation"
vec = embed_text(test_query)
if vec:
    search_body = {
        "vector": vec,
        "limit": 3,
        "with_payload": True
    }
    sr = requests.post(
        f"{QDRANT_URL}/collections/{COLLECTION}/points/search",
        json=search_body, timeout=15
    )
    results = sr.json().get("result", [])
    print(f"Test search: '{test_query}'")
    print(f"Top {len(results)} results:")
    for i, hit in enumerate(results):
        payload = hit.get("payload", {})
        score   = hit.get("score", 0)
        print(f"  [{i+1}] score={score:.3f} | {payload.get('source_name')} | {payload.get('source_article_title','')[:50]}")
    print()

if vectors_count > 0:
    print("✅ Qdrant knowledge base is ready — pipeline will now find evidence for claims!")
else:
    print("❌ No vectors found — check Cell 4 for errors.")